# 6. Interactive diagrams (ipyelk)

`longeron.diagrams` renders models as **interactive ELK diagrams** in
JupyterLab: pan/zoom, collapsible hierarchy, and click-selection that
maps back to model elements. Layout runs in the browser (elkjs via
[ipyelk](https://github.com/jupyrdf/ipyelk), vendored in `vendor/ipyelk`
with local fixes), so the cells below execute headlessly and the
diagrams lay themselves out when a frontend attaches.

Three views, one dispatcher:

| Function | Shows |
|---|---|
| `structure_diagram` | packages, defs (attribute compartments), nested usages, specialization/typing/connection edges |
| `state_diagram` | hierarchical states, entry markers, labeled transitions |
| `action_diagram` | the succession control-flow graph the interpreter executes |
| `diagram` | picks a view from the element's kind |

**You will learn how to:**

- draw structure, state, and action views of a model;
- resolve browser click-selections back to model elements
  (`on_select`);
- use the diagram toolbar: compact icon buttons plus a live search
  box that highlights matches (`longeron.toolbar`);
- export the same views to SVG/PNG headlessly (`longeron.render`);
- replay a simulation or an action execution over its diagram
  (`longeron.replay`).

**Prerequisites:** a source checkout with the vendored ipyelk installed
(`pip install -e vendor/ipyelk`, or `pixi run lab`). The replay section
needs the `replay` extra. On a static page the widgets show a text
placeholder; run this notebook in JupyterLab for the live versions.

In [ ]:
import longeron
from longeron import diagrams

## Structure: definitions, compartments, and relationships

What to look for in the diagram below: blue definition boxes with
attribute compartments, green usage boxes, and distinct edge styles for
specialization, typing, and connection.

In [ ]:
model = longeron.loads("""
package Rover {
    part def Wheel { attribute diameter : Real = 0.3; }
    part def Motor { attribute torque : Real = 4.2; }
    abstract part def Platform { attribute mass : Real; }

    part def Rover :> Platform {
        attribute mass : Real :>> Platform::mass = 18.0;
        part wheels : Wheel[6];
        part drive : Motor;
    }

    part mission {
        part rover : Rover;
        part lander;
        connect rover to lander;
    }
}
""")
diagrams.structure_diagram(model)

Blue boxes are definitions («part def») with their attributes as
compartment rows, and green boxes are usages. Solid blue arrows are
specializations, dashed green arrows are typings, and plain edges are
connections.

**Click a node** and the selection is a qualified name. The cell below
simulates a browser click and reads the selected elements back:

In [ ]:
structure = diagrams.structure_diagram(model)

selected: list = []
diagrams.on_select(structure, model, selected.extend)

# simulate a browser click (this is what selecting in the UI does):
structure.view.selection.ids = ["Rover::Rover"]
[f"{type(e).__name__}({e.kind}) {e.qualified_name}" for e in selected]

## The toolbar: compact controls and search

Hover over any diagram and a toolbar slides in along the top edge:
icon-only **Fit** / **Center** / **Toggle collapse** buttons (each with
a tooltip) and a **search box**. Typing highlights *every* node whose
name or qualified name contains the text (case-insensitive) with a
saturated outline, dims everything else, and shows a `matches/total`
count; the ✕ button (or emptying the box) restores the diagram.

Search is pure highlighting -- it never touches the selection, so
`on_select` callbacks cannot fire from it. It is also scriptable: the
cell below drives the same tool programmatically (note `selected` stays
unchanged). Pass `toolbar=False` to any view for ipyelk's stock
toolbar.

In [ ]:
from longeron import toolbar

finder = structure.get_tool(toolbar.DiagramSearch)
clicks_before = len(selected)
finder.query = "wheel"  # highlights Rover::Wheel and every wheel usage
print(f"{finder.match_count} of {finder.total_count} elements match")
print("on_select callbacks fired by searching:", len(selected) - clicks_before)
finder.query = ""  # clear: every highlight is removed

## State machines

What to look for in the diagram below: the black entry marker, the
nested states inside `playing`, and the time trigger labeled
`after 3600.0`.

In [ ]:
machine_model = longeron.loads("""
package Machines {
    state def Player {
        entry; then stopped;
        state stopped;
        transition first stopped accept play then playing;
        state playing {
            entry; then normal;
            state normal;
            transition first normal accept fast then fastForward;
            state fastForward;
            transition first fastForward accept fast then normal;
        }
        transition first playing accept stop then stopped;
        transition first playing accept after 3600.0 then stopped;
    }
}
""")
diagrams.state_diagram(machine_model.find("Machines::Player"))

The same model drives the simulator, so the diagram and the execution
can never disagree. The trace below walks the states the diagram just
drew:

In [ ]:
sim = longeron.Interpreter(machine_model).simulate(
    "Machines::Player", events=["play", "fast", "stop"]
)
[str(step) for step in sim.trace]

## Action flow: the graph the interpreter actually executes

What to look for in the diagram below: the decide node `d1` routing to
`inspect` or `abort`, and the two branches re-joining at `done`. The
follow-up cell runs both branches.

In [ ]:
flow_model = longeron.loads("""
package Ops {
    action def Deploy {
        in tested : Boolean;
        out log : String;
        assign log := "";
        action build { assign log := log + "build>"; }
        action inspect { assign log := log + "inspect>"; }
        action ship { assign log := log + "ship"; }
        action abort { assign log := log + "ABORT"; }
        first start then build;
        first build then d1;
        decide d1;
        if tested then inspect;
        else abort;
        first inspect then ship;
        first ship then done;
        first abort then done;
    }
}
""")
deploy = flow_model.find("Ops::Deploy")
diagrams.action_diagram(deploy)

In [ ]:
interp = longeron.Interpreter(flow_model)
print("tested=true: ", interp.run_action(deploy, inputs={"tested": True}).outputs)
print("tested=false:", interp.run_action(deploy, inputs={"tested": False}).outputs)

## One dispatcher, and a full example

`diagrams.diagram(...)` picks the right view from the element kind. Try
it on the drone example shipped with the repo: a state machine first,
then the whole model, which falls back to the structure view.

In [ ]:
drone = longeron.load("../examples/drone.sysml")
diagrams.diagram(drone.find("Drone::FlightStates"))

In [ ]:
diagrams.diagram(drone)  # falls back to the structure view

## Exporting images

The same views render headlessly to SVG/PNG. `longeron.render` runs the
vendored elkjs layout in a node subprocess instead of the browser, so
no frontend is needed. PNG conversion needs the native cairo library
and skips itself when cairo is missing.

In [ ]:
import tempfile
from pathlib import Path

from longeron import render

out = Path(tempfile.mkdtemp())
render.to_svg(diagrams.state_diagram(machine_model.find("Machines::Player")), out / "player.svg")
print((out / "player.svg").stat().st_size, "bytes of SVG")

try:
    render.to_png(drone.find("Drone::FlightStates"), out / "flight.png")
    print("PNG written:", out / "flight.png")
except Exception as err:  # cairosvg needs the native cairo library
    print("PNG skipped:", err)

> **Note** -- ipyelk is vendored (`vendor/ipyelk`, BSD-3-Clause) and
> installed editable so it can be patched as needed. Every patch is
> marked `LOCAL PATCH` and catalogued in
> `vendor/ipyelk/README.vendor.md`: headless-safe pipeline scheduling
> (no `RuntimeError: no running event loop` outside Jupyter),
> browser round-trip fixes, and a prebuilt JupyterLab extension built
> from the patched TypeScript sources.

## Replaying a simulation

`longeron.replay` animates an execution trace over the exported state
diagram (needs the `replay` extra: `pip install "longeron[replay]"`).
`replay_widget` simulates the machine with the same event protocol as
`Interpreter.simulate` (names send events, numbers advance the clock),
bakes the diagram to SVG, and replays the recorded timeline in the
browser.

What to look for when you press play: active states light up green
(composite ancestors in a lighter tint), fired transitions pulse
orange, and the controls play and scrub through sim time -- or through
the step index when no time passes at all. The robot below runs a
30-second cleaning mission, and something moves every few seconds.

In [ ]:
from longeron import replay

robot_model = longeron.loads("""
package Robots {
    state def CleaningRobot {
        entry; then docked;
        state docked;
        transition first docked accept start then cleaning;
        state cleaning {
            entry; then sweeping;
            state sweeping;
            transition first sweeping accept after 6.0 then mopping;
            state mopping;
            transition first mopping accept after 4.0 then sweeping;
        }
        transition first cleaning accept after 15.0 then charging;
        state charging;
        transition first charging accept after 8.0 then docked;
    }
}
""")

replay.replay_widget(
    longeron.Interpreter(robot_model), "Robots::CleaningRobot", events=["start", 30.0]
)

### Replaying an action execution

The same widget replays **action** executions over the action diagram
(`replay_widget` auto-detects action definitions; `kind="action"`
forces it). The axis is the step index.

What to look for below: the currently-executing named step lights up,
traversed successions pulse (routed through decide/merge nodes), and
the readout line under the controls follows the scalar env values.
Watch `log` grow step by step.

In [ ]:
replay.replay_widget(interp, deploy, inputs={"tested": True})